In [7]:
# !git clone https://github.com/PieceOfBulka/gp5.git
# %cd gp5
# !git checkout miftakhutdinov-bert
# %cd src

fatal: destination path 'gp5' already exists and is not an empty directory.
/kaggle/working/gp5
Already on 'miftakhutdinov-bert'
Your branch is up to date with 'origin/miftakhutdinov-bert'.
/kaggle/working/gp5/src


In [8]:
!pip install -r ../requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 40.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 33.5 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=593c3ed63a9a51058b6bcefe74fbedfd81cc3189199a66c8f0662e0b86a5aa20
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all 

In [47]:
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml/'
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
os.environ['CLEARML_API_ACCESS_KEY'] = 'FQE64YU1K13WA9ZBNT5XO6QG9PL8V6'
os.environ['CLEARML_API_SECRET_KEY'] = 'C6TLawyrvQTGlPWVI_kJes4CbFDbnjLgbL22j2axldSQouzzloM7FAT-ug4maSL4vw4'
os.environ['CLEARML_PROJECT'] = 'HSE-GP5'
os.environ['CLEARML_TASK'] = 'Bert'
os.environ['CLEARML_LOG_MODEL'] = 'True'

In [66]:
import os
import ast
from collections import Counter
import re
from tqdm.auto import tqdm

import dotenv
dotenv.load_dotenv()


import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch import nn

from transformers import (BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments)
from transformers.integrations import ClearMLCallback



from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score, hamming_loss)


from clearml import (Dataset, Task)

In [49]:
# task = Task.init(project_name="HSE-GP5", task_name='Bert')

configuration={'max_features':5000, 'ngram_range':(1,2), 'stop_words':'english', 'model_state':42, 'test_size':0.2, 'val_size': 0.1, 'split_state':42}
# task.connect(configuration)

In [50]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
device

'cuda'

In [51]:
dataset = Dataset.get(dataset_project="HSE-GP5", dataset_name='final_lyrics_dataset')
dataset_path = dataset.get_local_copy()

df = pd.read_csv(os.path.join(dataset_path,'final_dataset.csv'),encoding='utf-8',escapechar="\\")
df = df[['lyrics','genres']].dropna()
df['genres'] = df['genres'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genres'])
# task.upload_artifact(name="genre_classes", artifact_object=set(mlb.classes_))

2026-06-16 08:12:03,633 - clearml - INFO - Dataset.get() did not specify alias. Dataset information will not be automatically logged in ClearML Server.


In [52]:
XtrainVal, Xtest, ytrainVal, ytest = train_test_split(df['lyrics'], y, test_size=configuration['test_size'], random_state=configuration['split_state'])
Xtrain, Xval, ytrain, yval = train_test_split(XtrainVal, ytrainVal, test_size=configuration['val_size'], random_state=configuration['split_state'])

In [128]:
tokenizer = BertTokenizerFast.from_pretrained('google-bert/bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=len(mlb.classes_), problem_type='multi_label_classification').to(device) # https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForSequenceClassification, https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google-bert/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-bert/bert-base-uncased/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google-bert/bert-base-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google-bert/bert-base-uncased/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
INFO:h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [129]:
class BertDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, text, labels):
        self.embeddings = tokenizer(text=text, truncation=True, padding='max_length', return_tensors='pt') # https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PythonBackend.__call__, pytorch, обрезаем текст если не влезает
        self.labels = torch.tensor(labels, dtype=torch.float32) # переводим тензоры y в float32
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.embeddings.items()}
        item['labels'] = self.labels[idx]
        return item

In [130]:
train_dataset = BertDataset(text=Xtrain.tolist(), labels=ytrain, tokenizer=tokenizer)
val_dataset = BertDataset(text=Xval.tolist(), labels=yval, tokenizer=tokenizer)
test_dataset = BertDataset(text=Xtest.tolist(), labels=ytest, tokenizer=tokenizer)

In [53]:
def evaluate(preds):
    logits, labels = preds
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)
    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(labels, preds) # доля неверно предсказанных лейблов по классам
    }

In [132]:
trainer = Trainer(model, 
                args=TrainingArguments(
                    output_dir='./bert', # куда сохранять ЧП во время обучения
                    num_train_epochs=5, # Число эпох обучения
                    learning_rate=1e-5, # lr, меньше т.к. файнтуним
                    weight_decay=0.01, # L2-регуляризация
                    per_device_train_batch_size=32, # Какого размера передаем батч на каждый шаг
                    per_device_eval_batch_size=32,
                    warmup_steps=500, # Распределение шагов до того, как модель придет к 1e-3
                    eval_strategy='epoch', # Как часто считаем метрики на валидации
                    save_strategy='epoch', # Как часто сохранять ЧП
                    load_best_model_at_end=True, # Загружать итоговую модель не последнюю, а лучшую по метрике
                    metric_for_best_model='f1_micro', # Определяем f1_micro как метрику, по которой определяем лучшую версию модели f1-micro — считаем TP, FP, FN по всем классам, а потом суммируем (взвешенный вариант f1-macro)
                    logging_steps=20 # Как часто логируем
                    ),
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                compute_metrics=evaluate,
                callbacks=[ClearMLCallback()] # Коллбек для логирования сразу в клирмл https://huggingface.co/docs/transformers/main_classes/callback#transformers.integrations.ClearMLCallback
            ) # https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer

In [133]:
trainer.train()

Parameters must be of builtin type (Transformers_5/accelerator_config[AcceleratorConfig])
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning:

Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.



Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,1.355186,1.303183,0.068862,0.047693,0.387785,8.377700,59.563000,0.955000
2,1.140859,1.092429,0.055605,0.012179,0.103099,8.344800,59.797000,0.959000
3,1.001478,0.926640,0.069023,0.002847,0.043563,8.304000,60.091000,0.963000
4,0.813149,0.753974,0.000000,0.000000,0.037356,8.303600,60.094000,0.963000
5,0.651817,0.578169,0.000000,0.000000,0.037356,8.312500,60.030000,0.962000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning:

Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning:

Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=355, training_loss=1.0311301164224114, metrics={'train_runtime': 1329.7538, 'train_samples_per_second': 16.86, 'train_steps_per_second': 0.267, 'total_flos': 5904299254517760.0, 'train_loss': 1.0311301164224114, 'epoch': 5.0})

In [134]:
predictions = trainer.predict(test_dataset)
test_metrics = evaluate((predictions.predictions, predictions.label_ids))

logger = task.get_logger()
for key, value in test_metrics.items():
    logger.report_scalar(title='Test Metrics', series=key, value=value, iteration=0)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning:

Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.



In [135]:
task.upload_artifact(name='bert_model', artifact_object='./bert')
task.close()

In [54]:
task_cnn = Task.init(project_name="HSE-GP5", task_name='TextCNN')
cnn_config = {
    'vocab_size': 20000,
    'max_len': 512,
    'embed_dim': 128, # размер эмбединга
    'num_filters': 100,
    'filter_sizes': (3, 4, 5), # Берем разные окна, чтобы захватить текст разного размера
    'lr': 1e-3,
    'batch_size': 64,
    'num_epochs': 10
}
task_cnn.connect(cnn_config)

{'vocab_size': 20000,
 'max_len': 512,
 'embed_dim': 128,
 'num_filters': 100,
 'filter_sizes': (3, 4, 5),
 'lr': 0.001,
 'batch_size': 64,
 'num_epochs': 10}

In [55]:
def tokenize(text):
    return re.findall(r"\w+", text.lower()) # Поиск последовательности букв = слов

counter = Counter()
for text in Xtrain:
    counter.update(tokenize(text)) # Частота слов

vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common(cnn_config['vocab_size']))} # Берем 20000 самых популярных слов
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def encode(text, max_len=512):
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens[:max_len]] # Обрезаем текст до 512 символов и токенизируем текст
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

In [56]:
class CNNDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, max_len=300):
        self.texts = [encode(t, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx]), torch.tensor(self.labels[idx], dtype=torch.float32)

In [61]:
train_ds = CNNDataset(Xtrain.tolist(), ytrain, cnn_config['max_len'])
val_ds = CNNDataset(Xval.tolist(), yval, cnn_config['max_len'])
test_ds = CNNDataset(Xtest.tolist(), ytest, cnn_config['max_len'])

train_loader = DataLoader(train_ds, batch_size=cnn_config['batch_size'])
val_loader = DataLoader(val_ds, batch_size=cnn_config['batch_size'])
test_loader = DataLoader(test_ds, batch_size=cnn_config['batch_size'])

In [63]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim, num_filters, filter_sizes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0) # айдишник паддинг токена — 0
        
        self.conv3 = nn.Conv1d(embed_dim, num_filters, kernel_size=3)
        self.conv4 = nn.Conv1d(embed_dim, num_filters, kernel_size=4)
        self.conv5 = nn.Conv1d(embed_dim, num_filters, kernel_size=5)

        self.dropout = nn.Dropout(0.5)
        
        self.fc = nn.Linear(num_filters * 3, num_classes) # 3 вектора склеенные между собой маппим в итоговые классы

    def forward(self, x):
        
        x = self.embedding(x) # batch, seq_len, emb_dim — получаем эмбендинг нашего объекта x
        x = x.transpose(1, 2) # batch, emb_dim, seq_len — в соответсвии с тем, как мы подготовили в модели

        out3 = torch.relu(self.conv3(x)) # Применяем три свертки
        out4 = torch.relu(self.conv4(x))
        out5 = torch.relu(self.conv5(x))

        pl3 = torch.max(out3, dim=2)[0] # Макспулим
        pl4 = torch.max(out4, dim=2)[0]
        pl5 = torch.max(out5, dim=2)[0]

        united = torch.cat([pl3, pl4, pl5], dim=1)
        united = self.dropout(united)

        logits = self.fc(united) # Применяем линейный слой поверх
        return logits

In [64]:
model_cnn = TextCNN(vocab_size=len(vocab), num_classes=len(mlb.classes_), embed_dim=cnn_config['embed_dim'], num_filters=cnn_config['num_filters'], filter_sizes=cnn_config['filter_sizes']).to(device)
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=cnn_config['lr'])
criterion = nn.BCEWithLogitsLoss()

In [67]:
logger = task_cnn.get_logger()
best_f1_micro = -1
global_step = 0

for epoch in range(cnn_config['num_epochs']):
    
    model_cnn.train()
    total_loss = 0
    
    for texts, labels in tqdm(train_loader, desc=f'epoch {epoch+1}'):
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model_cnn(texts)
        
        loss = criterion(logits, labels)
        loss.backward()
        
        optimizer.step()
        total_loss += loss.item()

        logger.report_scalar(title='Train', series='loss', value=loss.item(), iteration=global_step)
        global_step += 1

    mean_train_loss = total_loss / len(train_loader)

    model_cnn.eval()
    all_logits, all_labels = [], []
    
    with torch.no_grad():
        for texts, labels in val_loader:
            texts = texts.to(device)
            logits = model_cnn(texts)
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    metrics = evaluate((all_logits, all_labels))

    logger.report_scalar(title='Val', series='train_loss', value=mean_train_loss, iteration=epoch)
    
    for key, value in metrics.items():
        logger.report_scalar(title='Val Metrics', series=key, value=value, iteration=epoch)

    if metrics['f1_micro'] > best_f1_micro:
        best_f1_micro = metrics['f1_micro']
        torch.save(model_cnn.state_dict(), 'best_textcnn.pt')

    print(f'epoch {epoch+1} train_loss: {mean_train_loss:.4f}')

epoch 1:   0%|          | 0/71 [00:00<?, ?it/s]

epoch 1 train_loss: 0.1199


epoch 2:   0%|          | 0/71 [00:00<?, ?it/s]

epoch 2 train_loss: 0.1156


epoch 3:   0%|          | 0/71 [00:00<?, ?it/s]

...

epoch 8 train_loss: 0.0922


epoch 9:   0%|          | 0/71 [00:00<?, ?it/s]

epoch 9 train_loss: 0.0876


epoch 10:   0%|          | 0/71 [00:00<?, ?it/s]

epoch 10 train_loss: 0.0837


In [68]:
model_cnn.load_state_dict(torch.load('best_textcnn.pt'))
model_cnn.eval()

all_logits, all_labels = [], []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        logits = model_cnn(texts)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())

In [69]:
all_logits = np.concatenate(all_logits)
all_labels = np.concatenate(all_labels)
test_metrics = evaluate((all_logits, all_labels))

for key, value in test_metrics.items():
    logger.report_scalar(title='Test Metrics', series=key, value=value, iteration=0)

task_cnn.upload_artifact(name='textcnn_model', artifact_object='best_textcnn.pt')
task_cnn.close()

In [70]:
test_metrics

{'f1_micro': 0.2138620572784274,
 'f1_macro': 0.03655448012010448,
 'hamming_loss': 0.03614673752123299}